# Transformer 架构与 Attention 机制：可运行学习笔记

**运行环境**：Python 3.12 标准库。Notebook 使用纯 Python 实现小矩阵前向计算，便于逐步核对形状、掩码和注意力权重；真实训练应使用支持自动求导的框架。

**学习路线**：向量相似度 → 缩放点积注意力 → 掩码 → 多头注意力 → 前馈网络与残差 → Encoder/Decoder 架构。

符号约定：序列长度为 $n$，模型维度为 $d_{\mathrm{model}}$，每个头的键维度为 $d_k$。本笔记中的矩阵以“行 = token，列 = 特征”表示。

## 1. 为什么需要 Attention

循环神经网络逐步处理序列；Attention 允许当前位置直接读取其他位置的信息。对输入 $X\in\mathbb R^{n\times d_{\mathrm{model}}}$，用可学习矩阵构造

$$Q=XW_Q,\qquad K=XW_K,\qquad V=XW_V.$$

其中 **Query（查询）**表示当前位置想找什么，**Key（键）**表示各位置可供匹配的特征，**Value（值）**表示匹配后被聚合的内容。自注意力（Self-Attention）的 $Q,K,V$ 来自同一序列；交叉注意力（Cross-Attention）的 $Q$ 来自解码器，$K,V$ 来自编码器输出。

In [1]:
import math
import random

random.seed(7)

def transpose(a):
    return [list(col) for col in zip(*a)]

def matmul(a, b):
    assert a and b and len(a[0]) == len(b)
    return [[sum(x * y for x, y in zip(row, col)) for col in transpose(b)] for row in a]

def add(a, b):
    assert len(a) == len(b) and len(a[0]) == len(b[0])
    return [[x + y for x, y in zip(ra, rb)] for ra, rb in zip(a, b)]

def softmax(row):
    m = max(row)
    exps = [math.exp(x - m) if x != -math.inf else 0.0 for x in row]
    total = sum(exps)
    assert total > 0, "每个查询至少需要一个可见的键"
    return [x / total for x in exps]

def shape(a):
    return (len(a), len(a[0])) if a else (0, 0)

X = [[1.0, 0.0, 0.5, 0.0],
     [0.0, 1.0, 0.0, 0.5],
     [0.5, 0.5, 1.0, 0.0]]
print("输入形状 (token, 特征):", shape(X))

输入形状 (token, 特征): (3, 4)


## 2. 缩放点积注意力

单头注意力为

$$\operatorname{Attention}(Q,K,V)=\operatorname{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}+M\right)V.$$

$QK^\top$ 的第 $(i,j)$ 项是第 $i$ 个查询与第 $j$ 个键的相似度；除以 $\sqrt{d_k}$ 可减轻维度增大时点积幅度过大、softmax 过于尖锐的问题。掩码 $M$ 在不允许访问的位置取 $-\infty$，其余位置为 $0$。softmax **按每个查询的一行**计算。

In [2]:
def attention(q, k, v, allowed=None):
    assert len(k) == len(v) and len(q[0]) == len(k[0])
    dk = len(k[0])
    scores = [[s / math.sqrt(dk) for s in row] for row in matmul(q, transpose(k))]
    if allowed is not None:
        assert len(allowed) == len(q) and all(len(row) == len(k) for row in allowed)
        scores = [[score if ok else -math.inf for score, ok in zip(row, mask)]
                  for row, mask in zip(scores, allowed)]
    weights = [softmax(row) for row in scores]
    output = matmul(weights, v)
    return output, weights

out, weights = attention(X, X, X)
print("分数矩阵形状:", (len(X), len(X)))
print("注意力权重:")
for row in weights:
    print([round(x, 3) for x in row], "行和 =", round(sum(row), 6))
print("输出形状:", shape(out))
assert all(abs(sum(row) - 1.0) < 1e-12 for row in weights)

分数矩阵形状: (3, 3)
注意力权重:
[0.414, 0.221, 0.365] 行和 = 1.0
[0.241, 0.45, 0.309] 行和 = 1.0
[0.326, 0.254, 0.419] 行和 = 1.0
输出形状: (3, 4)


## 3. 掩码：因果与填充

**因果掩码（Causal Mask）**只允许位置 $i$ 访问 $j\le i$，用于自回归生成，防止读到未来 token。**填充掩码（Padding Mask）**使真实 token 不读取填充位。掩码作用在 softmax **之前**；在 softmax 之后把权重清零会破坏行和为 1。

In [3]:
n = len(X)
causal = [[j <= i for j in range(n)] for i in range(n)]
causal_out, causal_weights = attention(X, X, X, causal)
print("因果掩码:", causal)
print("因果权重:")
for row in causal_weights:
    print([round(x, 3) for x in row])
assert all(causal_weights[i][j] == 0.0 for i in range(n) for j in range(i + 1, n))

# 假设第三个位置是填充位；这里只展示“键不可见”，实际还应排除填充查询的输出。
padding = [[j < 2 for j in range(n)] for _ in range(n)]
_, pad_weights = attention(X, X, X, padding)
assert all(row[2] == 0.0 for row in pad_weights)
print("填充键的权重:", [row[2] for row in pad_weights])

因果掩码: [[True, False, False], [True, True, False], [True, True, True]]
因果权重:
[1.0, 0.0, 0.0]
[0.349, 0.651, 0.0]
[0.326, 0.254, 0.419]
填充键的权重: [0.0, 0.0, 0.0]


## 4. 多头注意力（Multi-Head Attention）

第 $h$ 个头使用自己的 $W_Q^{(h)},W_K^{(h)},W_V^{(h)}$，分别计算 $\mathrm{head}_h=\operatorname{Attention}(XW_Q^{(h)},XW_K^{(h)},XW_V^{(h)})$。把各头输出拼接，再投影：

$$\operatorname{MultiHead}(X)=\operatorname{Concat}(\mathrm{head}_1,\ldots,\mathrm{head}_H)W_O.$$

通常 $d_{\mathrm{model}}=H d_k$；不同头可以学习不同的关系，但**不保证**每个头都有清晰可解释的语义。下面采用 2 头、模型维度 4、每头维度 2。权重随机初始化，只用于演示前向形状，未经过训练。

In [4]:
def rand_matrix(rows, cols, scale=0.3):
    return [[random.uniform(-scale, scale) for _ in range(cols)] for _ in range(rows)]

def concat_columns(blocks):
    return [sum((block[i] for block in blocks), []) for i in range(len(blocks[0]))]

class MultiHeadAttention:
    def __init__(self, d_model=4, heads=2):
        assert d_model % heads == 0
        self.d_model, self.heads, self.dk = d_model, heads, d_model // heads
        self.wq = [rand_matrix(d_model, self.dk) for _ in range(heads)]
        self.wk = [rand_matrix(d_model, self.dk) for _ in range(heads)]
        self.wv = [rand_matrix(d_model, self.dk) for _ in range(heads)]
        self.wo = rand_matrix(d_model, d_model)

    def __call__(self, x, allowed=None):
        parts, all_weights = [], []
        for h in range(self.heads):
            q, k, v = matmul(x, self.wq[h]), matmul(x, self.wk[h]), matmul(x, self.wv[h])
            part, weights = attention(q, k, v, allowed)
            parts.append(part)
            all_weights.append(weights)
        return matmul(concat_columns(parts), self.wo), all_weights

mha = MultiHeadAttention()
mha_out, head_weights = mha(X, causal)
print("多头输出形状:", shape(mha_out))
print("权重张量形状 (头, 查询, 键):", (len(head_weights), len(head_weights[0]), len(head_weights[0][0])))
assert shape(mha_out) == shape(X)

多头输出形状: (3, 4)
权重张量形状 (头, 查询, 键): (2, 3, 3)


## 5. 位置、前馈层、残差与归一化

单独的自注意力不包含顺序信息：若同时重排输入 token，输出也随之重排。因此需要位置编码。原始 Transformer 可使用正弦/余弦位置编码；现代模型也可能采用可学习位置向量或相对位置方法。

一个 Transformer 层还包含逐位置前馈网络 $\operatorname{FFN}(x)=W_2\phi(W_1x+b_1)+b_2$、残差连接和层归一化（Layer Normalization）。下面是 **Pre-LN** 结构：$H=X+\operatorname{MHA}(\operatorname{LN}(X))$，$Y=H+\operatorname{FFN}(\operatorname{LN}(H))$。

In [5]:
def positional_encoding(length, d_model):
    result = []
    for pos in range(length):
        row = []
        for i in range(d_model):
            angle = pos / (10000 ** (2 * (i // 2) / d_model))
            row.append(math.sin(angle) if i % 2 == 0 else math.cos(angle))
        result.append(row)
    return result

def layer_norm(x, eps=1e-5):
    rows = []
    for row in x:
        mean = sum(row) / len(row)
        var = sum((v - mean) ** 2 for v in row) / len(row)
        rows.append([(v - mean) / math.sqrt(var + eps) for v in row])
    return rows

class FeedForward:
    def __init__(self, d_model=4, hidden=8):
        self.w1 = rand_matrix(d_model, hidden)
        self.w2 = rand_matrix(hidden, d_model)

    def __call__(self, x):
        hidden = [[max(0.0, v) for v in row] for row in matmul(x, self.w1)]
        return matmul(hidden, self.w2)

class TransformerBlock:
    def __init__(self, d_model=4, heads=2, hidden=8):
        self.attn = MultiHeadAttention(d_model, heads)
        self.ffn = FeedForward(d_model, hidden)

    def __call__(self, x, allowed=None):
        a, weights = self.attn(layer_norm(x), allowed)
        h = add(x, a)
        y = add(h, self.ffn(layer_norm(h)))
        return y, weights

x_pos = add(X, positional_encoding(len(X), len(X[0])))
block = TransformerBlock()
y, _ = block(x_pos, causal)
print("输入 + 位置编码:", shape(x_pos))
print("Transformer Block 输出:", shape(y))
assert shape(y) == shape(X)

输入 + 位置编码: (3, 4)
Transformer Block 输出: (3, 4)


## 6. 整体 Transformer 架构

| 部分 | 输入和操作 | 作用 |
| --- | --- | --- |
| Embedding + 位置编码 | token ID → 向量，再加入位置信息 | 提供词义与顺序 |
| Encoder Block | 无因果掩码的自注意力 → FFN；各子层有残差和归一化 | 双向融合输入上下文 |
| Decoder Block | 因果自注意力 → 对编码器输出做交叉注意力 → FFN | 只读已生成 token，并读取输入序列 |
| 输出层 | 隐状态 → 词表 logits → softmax | 给出下一个 token 的概率 |

**Encoder-only**（如双向表示模型）适合理解/分类，通常可双向读取；**Decoder-only**（自回归语言模型）使用因果掩码预测下一个 token；**Encoder–Decoder** 适合输入到输出的序列转换。架构选择取决于任务和训练目标，不是注意力公式不同。

计算量：全注意力需要 $n\times n$ 分数矩阵，时间和注意力矩阵内存随序列长度近似按 $O(n^2)$ 增长（固定模型维度时）。长序列可考虑窗口、稀疏或其他高效注意力设计。

## 7. 学习检查与练习

1. 将因果掩码去掉，比较第一个位置的权重：它现在是否能读取未来位置？
2. 把第三个 token 设为填充，解释为什么不仅要屏蔽“作为键”的填充位，还要忽略“作为查询”的填充输出。
3. 把 $d_{\mathrm{model}}$ 改为 6、头数改为 3，检查每一层形状。
4. 手算两个二维 token 的 $QK^\top/\sqrt{d_k}$、softmax 和加权和，与代码结果核对。
5. 训练模型时，随机初始化的矩阵需要通过损失函数反向传播更新；本 Notebook 的纯标准库代码**只实现前向传播**，不包含自动求导、参数训练、Dropout 或真实词表。

**排错顺序**：先检查矩阵形状 → 每行注意力权重和是否为 1 → 被遮挡位置权重是否为 0 → 位置编码和残差后的形状 → 训练时再看损失、梯度与数据泄漏。